# 🔬 Notebook 05: CAMELYON16 — Análisis de Whole Slide Images (WSI) y Pipeline MIL
**Dataset:** CAMELYON16 — Lymph Node Metastasis Detection Challenge  
**Fuente:** Grand Challenge (Radboud University Medical Center & UMC Utrecht) — https://camelyon16.grand-challenge.org/  
**Artículo de Referencia:** Bejnordi et al. (2017) — *Diagnostic Assessment of Deep Learning Algorithms for Detection of Lymph Node Metastases in Women With Breast Cancer* (JAMA, 318(22):2199–2210)  
**Modalidad:** Histopatología Digital (H&E), Whole Slide Imaging (WSI) gigapíxel a 20x/40x  
**Paradigma:** Multiple Instance Learning (MIL) con Supervisión Débil a Nivel de Lámina  

---

## 📋 Resumen del Dataset y Enfoque de Análisis

El dataset **CAMELYON16** es el estándar de oro internacional en patología computacional para la detección automatizada de metástasis ganglionares axilares en cáncer de mama. A diferencia de problemas tradicionales de visión artificial con imágenes homogéneas de tamaño reducido (como MNIST, CIFAR o parches aislados), las láminas **WSI** son imágenes **gigapíxel** multirresolución (Aperio `.tif` / `.svs`) cuyas dimensiones sobrepasan los **100,000 × 100,000 píxeles** por lámina (~1 a 3 GB por archivo individual; ~700 GB para la cohorte completa de 399 láminas).

Para viabilizar el modelado computacional sin requerir terabytes de memoria en GPU, el flujo de trabajo moderno en patología digital adopta **Multiple Instance Learning (MIL)**:
1. **Detección Tisular:** Segmentación automatizada (umbralización Otsu sobre saturación de color) para aislar el tejido y descartar el fondo de vidrio.
2. **Tile & Patching:** Cuadriculado del tejido en parches canónicos de $224 \times 224$ píxeles a magnificación óptica de $20\times$ (aprox. $0.50\,\mu\text{m/px}$).
3. **Mapeo Espacial:** Registro exhaustivo de coordenadas espaciales `[level, x, y]` por cada parche tisular.
4. **Feature Extraction:** Transformación de cada parche en un vector latente denso (embedding de 768 a 1024 dimensiones) mediante Vision Transformers autosupervisados (p. ej., iBOT, UNI, Virchow, CTransPath) o ResNet50 preentrenada.
5. **Agregación MIL:** Modelos de agregación con mecanismo de atención (Attention-MIL, CLAM, TransMIL) que tratan a la lámina completa como una "bolsa" (*bag*) de parches (*instances*), prediciendo si la lámina es Normal ($Y=0$) o Tumoral ($Y=1$).

```text
WSI (Aperio .tif) → OpenSlide → Segmentación tisular (Otsu)
    ↓ Extracción parches 224×224px @ 20x
Coords: [level, x, y] por parche
    ↓ Feature Extractor (iBOT / UNI / ResNet50)
Embedding 1024-dim por parche
    ↓ MIL Aggregation
  Attention-MIL | TransMIL | CLAM
    ↓
Clasificación WSI: Normal (0) / Tumor (1)
```
---

In [1]:
# ==============================================================================
# IMPORTS, CONFIGURACIÓN DE ENTORNO Y FUNCIONES AUXILIARES
# ==============================================================================
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import xml.etree.ElementTree as ET
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuración global de visualización gráfica
plt.rcParams['figure.figsize'] = (16, 8)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.8

# Rutas de datos reales descargados en el workspace
COORDS_DIR = Path(r'c:\Universidad\Magister\20 Noviembre Reunion\data_samples\camelyon16\coords\CAMELYON16_FULL')
MASKS_DIR  = Path(r'c:\Universidad\Magister\20 Noviembre Reunion\data_samples\camelyon16\masks')
FIG_DIR    = Path('../figures/05_CAMELYON16')
TABLAS_DIR = Path('../tablas')

# Creación garantizada de carpetas de salida
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLAS_DIR.mkdir(parents=True, exist_ok=True)

def guardar_subplots_individuales(fig, nombre_base, fig_dir, dpi=150):
    """
    Exporta cada eje gráfico (subplot) de una figura como imagen individual
    utilizando la transformación de coordenadas y márgenes de resguardo.
    """
    axes = [ax for ax in fig.axes if ax.get_visible()]
    for i, ax in enumerate(axes):
        extent = ax.get_window_extent().transformed(fig.dpi_scale_trans.inverted())
        bbox = extent.expanded(1.2, 1.2)
        outpath = Path(fig_dir) / f'{nombre_base}_panel_{i+1}.png'
        fig.savefig(outpath, bbox_inches=bbox, dpi=dpi)
    print(f'✅ {len(axes)} paneles individuales guardados en: {Path(fig_dir).name}/')

print('✅ Entorno inicializado exitosamente.')
print(f'📁 Directorio Coordenadas: {COORDS_DIR.resolve()}')
print(f'📁 Directorio Máscaras:    {MASKS_DIR.resolve()}')
print(f'📁 Directorio Figuras:     {FIG_DIR.resolve()}')
print(f'📁 Directorio Tablas:      {TABLAS_DIR.resolve()}')

✅ Entorno inicializado exitosamente.
📁 Directorio Coordenadas: c:\Universidad\Magister\20 Noviembre Reunion\data_samples\camelyon16\coords\CAMELYON16_FULL
📁 Directorio Máscaras:    c:\Universidad\Magister\20 Noviembre Reunion\data_samples\camelyon16\masks
📁 Directorio Figuras:     c:\Universidad\Magister\20 Noviembre Reunion\figures\05_CAMELYON16
📁 Directorio Tablas:      c:\Universidad\Magister\20 Noviembre Reunion\tablas


---
### 📌 ¿Qué hace esta celda?
Configura el entorno de ejecución, importa las librerías matemáticas, de análisis tabular, de visualización y de procesamiento de documentos XML. Define las rutas absolutas y relativas a las fuentes de datos reales descargadas (`coords/CAMELYON16_FULL` y `masks/`), asegura la existencia de los directorios de exportación para figuras (`../figures/05_CAMELYON16`) y tablas Excel (`../tablas`), y define la rutina utilitaria `guardar_subplots_individuales` para la generación automática de paneles individuales con calidad editorial.

### ⚙️ ¿Cómo funciona?
- **`pathlib.Path`**: Manejo robusto y agnóstico de rutas del sistema de archivos, garantizando compatibilidad entre Windows y Linux.
- **`xml.etree.ElementTree`**: Parser nativo de alto rendimiento para procesar las anotaciones patológicas en formato XML de ASAP sin requerir dependencias externas pesadas.
- **`guardar_subplots_individuales()`**: Utiliza la matriz de transformación del lienzo (`fig.dpi_scale_trans.inverted()`) para calcular el *bounding box* exacto de cada eje (`ax.get_window_extent()`), expandiéndolo con un margen de seguridad del 20% (`bbox.expanded(1.2, 1.2)`) para preservar títulos y etiquetas sin solapamientos.

### 📊 Flujo de Inicialización
```text
┌─────────────────────────┐     ┌─────────────────────────┐     ┌─────────────────────────┐
│ Importar Librerías      │ ──> │ Validar Directorios     │ ──> │ Configurar Helper       │
│ (np, pd, plt, ET, Path) │     │ (coords, masks, tablas) │     │ Subplots Individuales   │
└─────────────────────────┘     └─────────────────────────┘     └─────────────────────────┘
```
---

In [2]:
# ==============================================================================
# CARGA Y ANÁLISIS EXHAUSTIVO DE COORDENADAS DE LAS 399 WSIs
# ==============================================================================
records = []
wsi_subdirs = sorted([d for d in COORDS_DIR.iterdir() if d.is_dir()])

for wsi_dir in wsi_subdirs:
    coords_file = wsi_dir / 'coords.npy'
    if coords_file.exists():
        coords = np.load(coords_file)
        is_tumor = 'Tumor' in wsi_dir.name or 'tumor' in wsi_dir.name
        is_test  = 'test' in wsi_dir.name.lower()
        records.append({
            'WSI': wsi_dir.name,
            'Tipo': 'Tumor' if is_tumor else ('Test' if is_test else 'Normal'),
            'Split': 'Test' if is_test else 'Train',
            'N_Parches': int(len(coords)),
            'Nivel_Resolucion': int(coords[0, 0]) if len(coords) > 0 else -1,
        })

df_coords = pd.DataFrame(records)

print(f'Total WSIs: {len(df_coords)}')
print(f'Total acumulado de parches: {df_coords["N_Parches"].sum():,} (224×224 px)\n')

# Agregación por Cohorte y Tipo histológico
df_resumen = df_coords.groupby(['Tipo', 'Split']).agg(
    N_WSI=('WSI', 'count'),
    Total_Parches=('N_Parches', 'sum'),
    Media_Parches=('N_Parches', lambda x: round(x.mean(), 1)),
    Mediana_Parches=('N_Parches', 'median'),
    Min_Parches=('N_Parches', 'min'),
    Max_Parches=('N_Parches', 'max'),
    Desv_Std=('N_Parches', lambda x: round(x.std(), 1))
).reset_index()

print('📊 Resumen cuantitativo por Cohorte y Tipo histológico:')
print(df_resumen.to_string(index=False))

# Guardar DataFrame detallado y resumen consolidado en Excel
excel_coords_path = TABLAS_DIR / 'camelyon16_coords_por_wsi.xlsx'
with pd.ExcelWriter(excel_coords_path, engine='openpyxl') as writer:
    df_coords.to_excel(writer, sheet_name='Detalle_399_WSI', index=False)
    df_resumen.to_excel(writer, sheet_name='Resumen_Cohortes', index=False)

print(f'\n✅ Excel guardado: {excel_coords_path.relative_to(Path.cwd().parent) if excel_coords_path.is_relative_to(Path.cwd().parent) else excel_coords_path}')

Total WSIs: 399
Total acumulado de parches: 3,177,267 (224×224 px)

📊 Resumen cuantitativo por Cohorte y Tipo histológico:
  Tipo Split  N_WSI  Total_Parches  Media_Parches  Mediana_Parches  Min_Parches  Max_Parches  Desv_Std
Normal Train    159        1126118         7082.5           6594.0          624        26297    4512.3
  Test  Test    130        1035959         7968.9           7280.5          842        24810    4880.1
 Tumor Train    110        1015190         9229.0           8154.5         1120        35766    5934.7

✅ Excel guardado: tablas/camelyon16_coords_por_wsi.xlsx


---
### 📌 ¿Qué hace esta celda?
Itera sobre los 399 directorios de Whole Slide Images (WSIs) en el dataset completo de CAMELYON16 (`coords.npy`), carga los arreglos de coordenadas tridimensionales `[nivel, x, y]`, clasifica cada lámina según su condición histológica (Normal, Tumor o Test set) y partición de datos (Train vs. Test), computa métricas de centralidad y dispersión de la carga de parches, y exporta dos hojas de cálculo estructuradas a un archivo Excel (`../tablas/camelyon16_coords_por_wsi.xlsx`).

### ⚙️ ¿Cómo funciona?
- **Lectura binaria `np.load()`**: Carga rápida de los arreglos estructurados de NumPy que contienen las coordenadas de los $3,177,267$ parches extraídos a magnificación canónica $20\times$.
- **Indexación y Metadatos**:
  - `Tipo`: Discrimina entre láminas de tejido benigno (`Normal`: 159 en Train), tejido metastásico (`Tumor`: 110 en Train) y el conjunto ciego de evaluación (`Test`: 130).
  - `N_Parches`: Cuantifica el número total de parches tisulares válidos (filtrados previamente por umbralización Otsu, descartando áreas vacías de vidrio).
  - `Nivel_Resolucion`: Registra el nivel piramidal de extracción ($17$).
- **Exportación Excel multihola (`openpyxl`)**: Genera la hoja `Detalle_399_WSI` con el listado completo de láminas y la hoja `Resumen_Cohortes` con agregaciones cuantitativas (conteo, suma, media, mediana, mínimo, máximo y desviación estándar).

### 📊 Flujo de Ingesta y Agregación
```text
┌─────────────────────────────────┐
│ Directorio CAMELYON16_FULL      │
│ (399 carpetas WSI)              │
└────────────────┬────────────────┘
                 │
                 ▼ np.load('coords.npy')
┌─────────────────────────────────┐
│ Arreglos [nivel, x, y]          │
│ • 159 Normales  (1,126,118 p.)  │
│ • 110 Tumorales (1,015,190 p.)  │
│ • 130 Test      (1,035,959 p.)  │
└────────────────┬────────────────┘
                 │
                 ▼ Agregación Pandas & Excel
┌─────────────────────────────────┐
│ camelyon16_coords_por_wsi.xlsx  │
│ • Sheet 1: Detalle_399_WSI      │
│ • Sheet 2: Resumen_Cohortes     │
└─────────────────────────────────┘
```
---

In [3]:
# ==============================================================================
# VISUALIZACIÓN: DISTRIBUCIÓN DE PARCHES POR WSI Y TARJETA RESUMEN
# ==============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={'width_ratios': [2.2, 1]})

# Panel 1: Gráfico de barras ordenado por cantidad de parches
df_sorted = df_coords.sort_values('N_Parches', ascending=False).reset_index(drop=True)

color_map = {'Normal': '#2ecc71', 'Tumor': '#e74c3c', 'Test': '#3498db'}
bar_colors = [color_map.get(t, '#95a5a6') for t in df_sorted['Tipo']]

x_indices = np.arange(len(df_sorted))
ax1.bar(x_indices, df_sorted['N_Parches'], color=bar_colors, width=1.0, edgecolor='none', alpha=0.85)
ax1.set_title('Distribución de Parches Extraídos por WSI (399 Láminas CAMELYON16)', fontsize=13, fontweight='bold', pad=12)
ax1.set_xlabel('Índice de WSI (ordenadas de mayor a menor carga tisular)', fontsize=11)
ax1.set_ylabel('Número de Parches (224×224 px)', fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.35, axis='y')

# Leyenda con parches coloreados
legend_patches = [
    mpatches.Patch(color='#2ecc71', label=f'Normal (Train: 159 WSIs / {df_coords[df_coords["Tipo"]=="Normal"]["N_Parches"].sum():,} parches)'),
    mpatches.Patch(color='#e74c3c', label=f'Tumor (Train: 110 WSIs / {df_coords[df_coords["Tipo"]=="Tumor"]["N_Parches"].sum():,} parches)'),
    mpatches.Patch(color='#3498db', label=f'Test Set (130 WSIs / {df_coords[df_coords["Tipo"]=="Test"]["N_Parches"].sum():,} parches)')
]
ax1.legend(handles=legend_patches, loc='upper right', frameon=True, facecolor='white', framealpha=0.9, fontsize=9.5)

# Panel 2: Tarjeta ejecutiva con métricas clave
ax2.axis('off')
total_wsis = len(df_coords)
total_parches = df_coords['N_Parches'].sum()
mean_parches = df_coords['N_Parches'].mean()
median_parches = df_coords['N_Parches'].median()
max_parches = df_coords['N_Parches'].max()
min_parches = df_coords['N_Parches'].min()

card_text = (
    "📊 FICHA TÉCNICA DE COORDENADAS\n"
    "────────────────────────────────────────\n"
    f"• Total WSIs en dataset:      {total_wsis:,} láminas\n"
    f"  - Normales (Train):         159 ({159/total_wsis*100:.1f}%)\n"
    f"  - Tumorales (Train):        110 ({110/total_wsis*100:.1f}%)\n"
    f"  - Conjunto Test:            130 ({130/total_wsis*100:.1f}%)\n\n"
    f"• Total Parches Extraídos:    {total_parches:,}\n"
    f"• Tamaño de Parche:           224 × 224 px\n"
    f"• Nivel de Resolución:        20x (~0.50 µm/px)\n\n"
    f"• Media de parches / WSI:     {mean_parches:,.0f}\n"
    f"• Mediana de parches / WSI:   {median_parches:,.0f}\n"
    f"• Máximo en una WSI:          {max_parches:,}\n"
    f"• Mínimo en una WSI:          {min_parches:,}\n\n"
    "🎯 IMPLICACIÓN COMPUTACIONAL MIL:\n"
    "────────────────────────────────────────\n"
    "• Cada WSI constituye 1 'Bag' de instancias.\n"
    "• La alta variabilidad (624 a 35,766 parches)\n"
    "  justifica el uso de Attention-MIL o TransMIL,\n"
    "  que procesan bolsas de tamaño dinámico\n"
    "  sin necesidad de recorte uniforme artificial."
)

ax2.text(0.05, 0.5, card_text, fontsize=10.5, family='monospace', verticalalignment='center',
         bbox=dict(boxstyle='round,pad=1', facecolor='#f8f9fa', edgecolor='#ced4da', linewidth=1.5))

plt.tight_layout()

# Guardar figura combinada y paneles individuales
fig_path_dist = FIG_DIR / 'camelyon16_distribucion_parches.png'
fig.savefig(fig_path_dist, dpi=150, bbox_inches='tight')
print(f'✅ Figura combinada guardada: {fig_path_dist}')

guardar_subplots_individuales(fig, 'camelyon16_distribucion_parches', FIG_DIR)
plt.show()

✅ Figura combinada guardada: ..\figures\05_CAMELYON16\camelyon16_distribucion_parches.png
✅ 2 paneles individuales guardados en: 05_CAMELYON16/


---
### 📌 ¿Qué hace esta celda?
Genera una figura comparativa de alta resolución compuesta por dos paneles:
1. **Panel 1**: Gráfico de barras clasificado y ordenado decrecientemente que visualiza el número de parches de tejido por cada una de las 399 WSIs, distinguiendo por código de color semántico las láminas normales (verde), tumorales (rojo) y de evaluación (azul).
2. **Panel 2**: Tarjeta técnica ejecutiva que sintetiza las métricas clave de la cohorte histológica y expone las implicaciones metodológicas para los modelos de Multiple Instance Learning.
Exporta la figura combinada (`camelyon16_distribucion_parches.png`) y sus paneles individuales correspondientes.

### ⚙️ ¿Cómo funciona?
- **Ordenamiento dinámico**: `df_coords.sort_values('N_Parches', ascending=False)` ordena las láminas de mayor a menor superficie de tejido para evidenciar el espectro de tamaño biológico ganglionar.
- **Paleta Cromática Semántica**: Mapeo condicional `{Normal: #2ecc71, Tumor: #e74c3c, Test: #3498db}` que permite correlacionar de inmediato la condición diagnóstica con la masa celular.
- **Relevancia para MIL**: En Multiple Instance Learning, cada lámina constituye una *bag* (bolsa) con un número variable de instancias ($N_i \in [624, 35766]$). Esta asimetría extrema desaconseja métodos convolucionales fijos y fundamenta el uso de agregadores invariantes a la permutación basados en atención (*Attention-MIL*, *CLAM*, *TransMIL*).

### 📊 Flujo de Visualización de Carga Tisular
```text
┌─────────────────────────┐     ┌─────────────────────────┐     ┌─────────────────────────┐
│ df_coords ordenado      │ ──> │ Renderizado Eje 1:      │ ──> │ Renderizado Eje 2:      │
│ (399 WSIs descendente)  │     │ Bar Chart Cromático     │     │ Tarjeta Ficha Técnica   │
└─────────────────────────┘     └─────────────────────────┘     └─────────────────────────┘
                                             │                               │
                                             └───────────────┬───────────────┘
                                                             ▼
                                                fig.savefig() & guardar_subplots()
                                                • camelyon16_distribucion_parches.png
                                                • panel_1.png / panel_2.png
```
---

In [4]:
# ==============================================================================
# MAPA ESPACIAL DE COORDENADAS: TOP TUMOR VS TOP NORMAL WSI
# ==============================================================================
# Identificar WSI tumoral y normal con mayor densidad de tejido
top_tumor_wsi  = df_coords[df_coords['Tipo'] == 'Tumor'].sort_values('N_Parches', ascending=False).iloc[0]
top_normal_wsi = df_coords[df_coords['Tipo'] == 'Normal'].sort_values('N_Parches', ascending=False).iloc[0]

print(f'🔬 WSI Normal seleccionada: {top_normal_wsi["WSI"]} con {top_normal_wsi["N_Parches"]:,} parches')
print(f'🔬 WSI Tumor seleccionada:  {top_tumor_wsi["WSI"]} con {top_tumor_wsi["N_Parches"]:,} parches')

# Cargar arreglos de coordenadas [nivel, x, y]
coords_norm = np.load(COORDS_DIR / top_normal_wsi['WSI'] / 'coords.npy')
coords_tum  = np.load(COORDS_DIR / top_tumor_wsi['WSI'] / 'coords.npy')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Panel 1: WSI Normal
ax1.scatter(coords_norm[:, 1], coords_norm[:, 2], s=2.5, alpha=0.55, c='#2ecc71', edgecolors='none')
ax1.set_title(f'WSI Normal: {top_normal_wsi["WSI"]}\n({top_normal_wsi["N_Parches"]:,} parches de tejido)', 
              fontsize=12, fontweight='bold', pad=10)
ax1.set_xlabel('Coordenada Grid X (Nivel 17)', fontsize=11)
ax1.set_ylabel('Coordenada Grid Y (Nivel 17)', fontsize=11)
ax1.invert_yaxis()  # Origen superior izquierdo estándar en histopatología digital
ax1.set_aspect('equal')
ax1.grid(True, linestyle='--', alpha=0.35)

# Panel 2: WSI Tumoral
ax2.scatter(coords_tum[:, 1], coords_tum[:, 2], s=2.5, alpha=0.55, c='#e74c3c', edgecolors='none')
ax2.set_title(f'WSI Tumoral: {top_tumor_wsi["WSI"]}\n({top_tumor_wsi["N_Parches"]:,} parches de tejido)', 
              fontsize=12, fontweight='bold', pad=10)
ax2.set_xlabel('Coordenada Grid X (Nivel 17)', fontsize=11)
ax2.set_ylabel('Coordenada Grid Y (Nivel 17)', fontsize=11)
ax2.invert_yaxis()
ax2.set_aspect('equal')
ax2.grid(True, linestyle='--', alpha=0.35)

plt.suptitle('Reconstrucción Espacial 2D del Tejido a partir de Coordenadas de Parches', 
             fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()

# Guardar figura combinada y paneles individuales
fig_path_mapa = FIG_DIR / 'camelyon16_mapa_coords.png'
fig.savefig(fig_path_mapa, dpi=150, bbox_inches='tight')
print(f'✅ Mapa espacial guardado: {fig_path_mapa}')

guardar_subplots_individuales(fig, 'camelyon16_mapa_coords', FIG_DIR)
plt.show()

# Guardar tabla de límites geométricos en Excel
df_spatial_extents = pd.DataFrame([
    {
        'WSI': top_normal_wsi['WSI'],
        'Tipo': 'Normal',
        'N_Parches': int(top_normal_wsi['N_Parches']),
        'X_Min': float(coords_norm[:, 1].min()),
        'X_Max': float(coords_norm[:, 1].max()),
        'Y_Min': float(coords_norm[:, 2].min()),
        'Y_Max': float(coords_norm[:, 2].max()),
        'Rango_X': float(coords_norm[:, 1].max() - coords_norm[:, 1].min()),
        'Rango_Y': float(coords_norm[:, 2].max() - coords_norm[:, 2].min()),
    },
    {
        'WSI': top_tumor_wsi['WSI'],
        'Tipo': 'Tumor',
        'N_Parches': int(top_tumor_wsi['N_Parches']),
        'X_Min': float(coords_tum[:, 1].min()),
        'X_Max': float(coords_tum[:, 1].max()),
        'Y_Min': float(coords_tum[:, 2].min()),
        'Y_Max': float(coords_tum[:, 2].max()),
        'Rango_X': float(coords_tum[:, 1].max() - coords_tum[:, 1].min()),
        'Rango_Y': float(coords_tum[:, 2].max() - coords_tum[:, 2].min()),
    }
])
with pd.ExcelWriter(TABLAS_DIR / 'camelyon16_coordenadas_extremos.xlsx', engine='openpyxl') as writer:
    df_spatial_extents.to_excel(writer, sheet_name='Extremos_Espaciales', index=False)
print('✅ Tabla de extremos espaciales guardada en: tablas/camelyon16_coordenadas_extremos.xlsx')

🔬 WSI Normal seleccionada: Normal_076.tif con 26,297 parches
🔬 WSI Tumor seleccionada:  tumor_092.tif con 35,766 parches
✅ Mapa espacial guardado: ..\figures\05_CAMELYON16\camelyon16_mapa_coords.png
✅ 2 paneles individuales guardados en: 05_CAMELYON16/
✅ Tabla de extremos espaciales guardada en: tablas/camelyon16_coordenadas_extremos.xlsx


---
### 📌 ¿Qué hace esta celda?
Identifica y extrae las coordenadas de las láminas con mayor cantidad de tejido de cada clase principal (`tumor_092.tif` con 35,766 parches y `Normal_076.tif` con 26,297 parches). Proyecta un diagrama de dispersión (*scatter plot*) bidimensional en el espacio de la cuadrícula de extracción para reconstruir la topología macroscópica de los ganglios linfáticos axilares escaneados. Exporta la figura combinada (`camelyon16_mapa_coords.png`), los paneles individuales y un archivo Excel con los límites geométricos de ambas láminas (`camelyon16_coordenadas_extremos.xlsx`).

### ⚙️ ¿Cómo funciona?
- **Inversión del Eje Vertical (`ax.invert_yaxis()`)**: En microscopía digital y estándares de imágenes biomédicas, el origen de coordenadas $(0, 0)$ se sitúa en la esquina superior izquierda del portaobjetos. Invertir el eje $Y$ asegura la orientación espacial histopatológica correcta.
- **Proporción de Aspecto Isométrica (`ax.set_aspect('equal')`)**: Mantiene la escala 1:1 entre los ejes $X$ e $Y$, preservando la morfología y contornos reales del corte ganglionar.
- **Transparencia Alfa y Dispersión Subcelular**: El parámetro `alpha=0.55` y `s=2.5` permite apreciar variaciones en la densidad de empaquetamiento tisular y delimitar con precisión la cápsula ganglionar frente a zonas de grasa o tejido conjuntivo circundante.

### 📊 Flujo de Reconstrucción Espacial
```text
┌───────────────────────────────┐     ┌───────────────────────────────┐
│ WSI Normal: Normal_076.tif    │     │ WSI Tumor: tumor_092.tif      │
│ (26,297 parches)              │     │ (35,766 parches)              │
└───────────────┬───────────────┘     └───────────────┬───────────────┘
                │                                     │
                ▼ coords[:, 1] (X) vs. coords[:, 2] (Y)
┌─────────────────────────────────────────────────────────────────────┐
│ Proyección Scatter 2D con orientación anatómica (invert_yaxis)      │
│ • Reconstrucción morfológica del ganglio linfático                  │
│ • Verificación de cobertura homogénea de parches tisulares          │
└─────────────────────────────────┬───────────────────────────────────┘
                                  │
                                  ▼
┌─────────────────────────────────┴───────────────────────────────────┐
│ Exportación: camelyon16_mapa_coords.png + Paneles 1 y 2             │
│ Exportación Excel: camelyon16_coordenadas_extremos.xlsx             │
└─────────────────────────────────────────────────────────────────────┘
```
---

In [5]:
# ==============================================================================
# ANÁLISIS DE ANOTACIONES PATOLÓGICAS XML (MÁSCARAS TUMORALES)
# ==============================================================================
xml_files = sorted(list(MASKS_DIR.rglob('*.xml')))
print(f'Mascaras XML de tumor: {len(xml_files)}')

# Parsear UN archivo XML para mostrar estructura
if xml_files:
    sample_xml = xml_files[0]
    tree = ET.parse(sample_xml)
    root = tree.getroot()
    # Contar anotaciones (Annotations/Annotation)
    annotations = root.findall('.//Annotation')
    print(f'Ejemplo: {sample_xml.name}')
    print(f'  Anotaciones (regiones de tumor): {len(annotations)}')
    for ann in annotations[:3]:
        coords_ann = ann.findall('.//Coordinate')
        print(f'  Region con {len(coords_ann)} vertices')

# Estadísticas de tamaño y metadatos de todos los XMLs
mask_records = []
for f in sorted(xml_files):
    size_kb = round(f.stat().st_size / 1024, 1)
    mask_records.append({
        'Archivo': f.name,
        'WSI_ID': f.stem,
        'Tamano_KB': size_kb,
        'Formato': 'ASAP Annotations (XML)'
    })

df_masks = pd.DataFrame(mask_records)

print(f'\n📊 Estadisticas de almacenamiento de mascaras XML:')
print(f'  • Tamano medio por XML: {df_masks["Tamano_KB"].mean():.1f} KB')
print(f'  • Tamano minimo:        {df_masks["Tamano_KB"].min():.1f} KB')
print(f'  • Tamano maximo:        {df_masks["Tamano_KB"].max():.1f} KB')
print(f'  • Tamano total:         {df_masks["Tamano_KB"].sum()/1024:.2f} MB')

# Guardar en Excel con formato openpyxl
excel_masks_path = TABLAS_DIR / 'camelyon16_mascaras_xml.xlsx'
with pd.ExcelWriter(excel_masks_path, engine='openpyxl') as writer:
    df_masks.to_excel(writer, sheet_name='Archivos_XML', index=False)

print(f'✅ Excel guardado: {excel_masks_path}')

Mascaras XML de tumor: 111
Ejemplo: tumor_001.xml
  Anotaciones (regiones de tumor): 2
  Region con 855 vertices
  Region con 1657 vertices

📊 Estadisticas de almacenamiento de mascaras XML:
  • Tamano medio por XML: 515.1 KB
  • Tamano minimo:        4.9 KB
  • Tamano maximo:        3792.7 KB
  • Tamano total:         7.80 MB
✅ Excel guardado: ..\tablas\camelyon16_mascaras_xml.xlsx


---
### 📌 ¿Qué hace esta celda?
Realiza una inspección y caracterización exhaustiva de los 111 archivos XML de anotaciones patológicas suministrados en el dataset CAMELYON16 (`data_samples/camelyon16/masks/`). Analiza la estructura jerárquica del archivo representativo `tumor_001.xml` (desglosando anotaciones individuales, grupos y vértices polinomiales), cuantifica el volumen de almacenamiento de todas las máscaras y exporta una tabla técnica completa a Excel (`camelyon16_mascaras_xml.xlsx`).

### ⚙️ ¿Cómo funciona?
- **Formato ASAP (Automated Slide Analysis Platform)**: Las anotaciones de CAMELYON16 fueron delimitadas por patólogos expertos utilizando la herramienta ASAP. El esquema XML consta de nodos raíz `<ASAP_Annotations>`, agrupaciones `<AnnotationGroups>` y colecciones `<Annotations>` compuestas por polígonos cerrados de coordenadas continuas `<Coordinate Order="i" X="x" Y="y"/>`.
- **Resolución a Nivel de Píxel en Nivel 0**: Las coordenadas $(X, Y)$ del XML están referenciadas al nivel de máxima resolución (Nivel 0, $40\times$). Para mapear un parche $224 \times 224$ px a su condición tumoral, se realiza una prueba de punto en polígono (*Point-in-Polygon*, ej. algoritmo de ray casting o `shapely.geometry.Polygon`) o se rasteriza una máscara binaria a la resolución del parche.
- **Supervisión Fuerte vs. Débil**: Aunque estas máscaras permiten supervisión densa a nivel de píxel o parche, los métodos MIL modernos sólo requieren la etiqueta a nivel de lámina ($Y \in \{0, 1\}$), reservando estas anotaciones XML para evaluar la interpretabilidad de los mapas de atención generados por el modelo.

### 📊 Flujo de Parsing de Anotaciones XML
```text
┌─────────────────────────────────┐
│ Archivo XML (ej. tumor_001.xml) │
└────────────────┬────────────────┘
                 │
                 ▼ ElementTree.parse()
┌─────────────────────────────────┐
│ Nodo Raíz: <ASAP_Annotations>   │
│   └── <Annotations>             │
│         ├── <Annotation Name="_0" PartOfGroup="_0"> (855 vértices)
│         └── <Annotation Name="_1" PartOfGroup="_0"> (1657 vértices)
└────────────────┬────────────────┘
                 │
                 ▼ Aplicación en Modelado
┌─────────────────────────────────┬─────────────────────────────────┐
│ Evaluación de Interpretabilidad │ Exportación de Metadatos        │
│ • Mapas de calor de atención    │ • camelyon16_mascaras_xml.xlsx  │
│ • Superposición polígono tumor  │ • Tamaño medio: 515.1 KB        │
└─────────────────────────────────┴─────────────────────────────────┘
```
---

In [6]:
# ==============================================================================
# PIPELINE MIL: RESUMEN EJECUTIVO Y TABLA COMPARATIVA
# ==============================================================================
total_parches_sum = df_coords['N_Parches'].sum()
total_wsi_count = len(df_coords)

df_pipeline = pd.DataFrame([
    {
        'Etapa': '1. WSI Input', 
        'Descripcion': 'Whole Slide Image (Aperio .tif / .svs gigapixel)', 
        'Dimensiones': '~100,000 × 100,000 px por WSI',
        'Tamano': '~1-3 GB por WSI (399 WSIs: ~700 GB total)', 
        'Tool': 'OpenSlide / ASAP / CuPath'
    },
    {
        'Etapa': '2. Segmentacion Tisular', 
        'Descripcion': 'Detección de tejido y exclusión de vidrio vacío', 
        'Dimensiones': 'Máscara binaria a 5x (downsample 8x)',
        'Tamano': 'Reducción de ~80% de fondo vacío', 
        'Tool': 'OpenCV / scikit-image (Otsu threshold)'
    },
    {
        'Etapa': '3. Extracción Parches', 
        'Descripcion': 'Cuadriculado de tejido en parches 224×224 @ 20x', 
        'Dimensiones': '224 × 224 px por parche (~0.50 µm/px)',
        'Tamano': f'{total_parches_sum:,} parches totales (coords.npy)', 
        'Tool': 'Custom Extractor (coords.npy)'
    },
    {
        'Etapa': '4. Feature Extractor', 
        'Descripcion': 'Extracción de embedding denso por cada parche', 
        'Dimensiones': 'Embedding 768-dim (ViT) o 1024-dim (ResNet)',
        'Tamano': '~4.3 GB a ~11.9 GB embeddings H5', 
        'Tool': 'iBOT / UNI / Virchow / MoCo'
    },
    {
        'Etapa': '5. MIL Bag', 
        'Descripcion': 'Bolsa (Bag) = WSI completa (slide)', 
        'Dimensiones': 'Matriz N_parches × D_emb', 
        'Tamano': f'{total_wsi_count} bags (159 Normal, 110 Tumor, 130 Test)', 
        'Tool': 'CLAM / TransMIL / AB-MIL'
    },
    {
        'Etapa': '6. Clasificacion', 
        'Descripcion': 'Normal (0) vs. Tumor (1) bag-level', 
        'Dimensiones': 'Probabilidad P(Tumor|WSI) in [0, 1]', 
        'Tamano': 'AUC objetivo: >0.95', 
        'Tool': 'Attention-MIL / ABMIL / TransMIL'
    },
])

print('📋 PIPELINE MULTIPLE INSTANCE LEARNING (CAMELYON16):')
print(df_pipeline.to_string(index=False))

# Guardar en Excel con formato estructurado
excel_pipeline_path = TABLAS_DIR / 'camelyon16_pipeline_mil.xlsx'
with pd.ExcelWriter(excel_pipeline_path, engine='openpyxl') as writer:
    df_pipeline.to_excel(writer, sheet_name='Pipeline_MIL', index=False)

print(f'\n✅ Ficha del pipeline MIL guardada en: {excel_pipeline_path}')

📋 PIPELINE MULTIPLE INSTANCE LEARNING (CAMELYON16):
                 Etapa                                         Descripcion                          Dimensiones                                    Tamano                                 Tool
          1. WSI Input Whole Slide Image (Aperio .tif / .svs gigapixel)             ~100,000 × 100,000 px por WSI ~1-3 GB por WSI (399 WSIs: ~700 GB total)            OpenSlide / ASAP / CuPath
2. Segmentacion Tisular Detección de tejido y exclusión de vidrio vacío       Máscara binaria a 5x (downsample 8x)          Reducción de ~80% de fondo vacío OpenCV / scikit-image (Otsu threshold)
 3. Extracción Parches Cuadriculado de tejido en parches 224×224 @ 20x 224 × 224 px por parche (~0.50 µm/px)    3,177,267 parches totales (coords.npy)         Custom Extractor (coords.npy)
  4. Feature Extractor   Extracción de embedding denso por cada parche Embedding 768-dim (ViT) o 1024-dim (ResNet)          ~4.3 GB a ~11.9 GB embeddings H5          iBOT / UNI / 

---
### 📌 ¿Qué hace esta celda?
Consolida el diseño conceptual y operativo del pipeline completo de Multiple Instance Learning para CAMELYON16 en un resumen ejecutivo estandarizado. Especifica los requerimientos dimensionales, volumetrías de datos reales obtenidos en este análisis y el instrumental de software recomendado para cada una de las 6 etapas del flujo histopatológico digital. Exporta la tabla de especificaciones a un archivo Excel formateado (`camelyon16_pipeline_mil.xlsx`).

### ⚙️ ¿Cómo funciona?
- **Etapa 1 (WSI Input)**: Manejo de imágenes piramidales gigapíxel multirresolución mediante la librería `OpenSlide` (lectura selectiva de bloques sin descompresión global en RAM).
- **Etapa 2 (Segmentación)**: Detección de primer plano celular mediante umbralización de Otsu sobre el canal de saturación HSV, eliminando más del 80% de fondo vacío.
- **Etapa 3 (Parches)**: División del tejido en $3,177,267$ parches homogéneos de $224 \times 224$ px a $20\times$ (aprox. $0.50\,\mu\text{m/px}$).
- **Etapa 4 (Features)**: Extracción de vectores latentes (embeddings de 768 o 1024 dimensiones) utilizando Vision Transformers preentrenados de forma autosupervisada (*Foundation Models* de patología).
- **Etapa 5 (Bolsas MIL)**: Estructuración de láminas como bolsas de instancias para alimentar redes neuronales con pooling de atención invariante a la permutación.
- **Etapa 6 (Clasificación)**: Predicción diagnóstica binaria (Lámina Normal vs. Con Metástasis) optimizada mediante métricas ROC-AUC (>0.95 objetivo) y curvas FROC (*Free-Response Receiver Operating Characteristic*).

### 📊 Flujo del Pipeline MIL de Extremo a Extremo
```text
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│ 1. WSI Input │ ──> │ 2. Segmentar │ ──> │ 3. Parches   │
│ (~700 GB)    │     │ Tejido (Otsu)│     │ (3.17M tiles)│
└──────────────┘     └──────────────┘     └──────────────┘
                                                 │
                                                 ▼
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│ 6. Diagnóstico│ <── │ 5. Bolsas MIL│ <── │ 4. Features  │
│ (ROC-AUC)    │     │ (399 WSI bags│     │ (Embeddings) │
└──────────────┘     └──────────────┘     └──────────────┘
```
---

In [7]:
# ==============================================================================
# ESTRATEGIA MIL RECOMENDADA Y ACCESO A EMBEDDINGS EN KAGGLE
# ==============================================================================
print('=== ESTRATEGIA MIL RECOMENDADA ===')
print('''
En vez de procesar las WSI gigapixel directamente (~700 GB),
se usan embeddings pre-extraidos de alta fidelidad disponibles en Kaggle:

Opcion 1 - iBOT ViT-Base (COAD + CAMELYON16) [RECOMENDADA]:
  kaggle datasets download -d forcewithme/ibotvitbasecoadcamelyon16
  Tamano: ~4.3 GB | Embedding: 768-dim por parche | Vision Transformer

Opcion 2 - MoCo WideResNet (COAD + CAMELYON16):
  kaggle datasets download -d forcewithme/mocowideresnetcoadcamelyon16  
  Tamano: ~11.9 GB | Embedding: 2048-dim por parche | WideResNet-50

Flujo de entrenamiento:
  embeddings.h5 -> CLAM / TransMIL / AB-MIL -> ROC-AUC en test set (>0.95)
''')

# Tabla comparativa de opciones de embeddings exportada a Excel
df_estrategia_mil = pd.DataFrame([
    {
        'Opción': 'Opción 1: iBOT ViT-Base (Recomendada)',
        'Backbone': 'Vision Transformer (ViT-B/16)',
        'Dimensión Embedding': 768,
        'Tamaño Descarga': '~4.3 GB',
        'Repositorio Kaggle': 'forcewithme/ibotvitbasecoadcamelyon16',
        'Paradigma Preentrenamiento': 'Masked Image Modeling + Self-Distillation',
        'Uso Recomendado': 'TransMIL / CLAM-SB con atención Transformer'
    },
    {
        'Opción': 'Opción 2: MoCo v2 WideResNet',
        'Backbone': 'WideResNet-50-2',
        'Dimensión Embedding': 2048,
        'Tamaño Descarga': '~11.9 GB',
        'Repositorio Kaggle': 'forcewithme/mocowideresnetcoadcamelyon16',
        'Paradigma Preentrenamiento': 'Momentum Contrast (Contrastive Learning)',
        'Uso Recomendado': 'Benchmarking convolucional clásico con ABMIL'
    },
    {
        'Opción': 'Opción 3: Coordenadas Reales (Local)',
        'Backbone': 'Coordenadas 2D [level, x, y]',
        'Dimensión Embedding': '3 columnas (x, y, level)',
        'Tamaño Descarga': '11 MB (Disponible en workspace)',
        'Repositorio Kaggle': 'Local (data_samples/camelyon16/coords)',
        'Paradigma Preentrenamiento': 'Grid espacial de parches 224×224 px',
        'Uso Recomendado': 'Análisis de distribución espacial y filtrado de tejido'
    }
])

excel_estrategia_path = TABLAS_DIR / 'camelyon16_estrategia_mil.xlsx'
with pd.ExcelWriter(excel_estrategia_path, engine='openpyxl') as writer:
    df_estrategia_mil.to_excel(writer, sheet_name='Estrategias_MIL', index=False)

print(f'✅ Tabla de estrategia MIL guardada en: {excel_estrategia_path}')

=== ESTRATEGIA MIL RECOMENDADA ===

En vez de procesar las WSI gigapixel directamente (~700 GB),
se usan embeddings pre-extraidos de alta fidelidad disponibles en Kaggle:

Opcion 1 - iBOT ViT-Base (COAD + CAMELYON16) [RECOMENDADA]:
  kaggle datasets download -d forcewithme/ibotvitbasecoadcamelyon16
  Tamano: ~4.3 GB | Embedding: 768-dim por parche | Vision Transformer

Opcion 2 - MoCo WideResNet (COAD + CAMELYON16):
  kaggle datasets download -d forcewithme/mocowideresnetcoadcamelyon16  
  Tamano: ~11.9 GB | Embedding: 2048-dim por parche | WideResNet-50

Flujo de entrenamiento:
  embeddings.h5 -> CLAM / TransMIL / AB-MIL -> ROC-AUC en test set (>0.95)

✅ Tabla de estrategia MIL guardada en: ..\tablas\camelyon16_estrategia_mil.xlsx


---
### 📌 ¿Qué hace esta celda?
Expone la estrategia metodológica de investigación recomendada para el entrenamiento y experimentación de modelos MIL sobre CAMELYON16 sin incurrir en semanas de preprocesamiento gigapíxel ni requerir servidores masivos de almacenamiento. Identifica los repositorios de embeddings preextraídos disponibles en Kaggle (iBOT ViT-Base y MoCo WideResNet), detalla sus atributos de compresión y dimensión, define el protocolo de entrenamiento con arquitecturas de última generación (CLAM, TransMIL, ABMIL) y exporta la tabla de estrategias a Excel (`camelyon16_estrategia_mil.xlsx`).

### ⚙️ ¿Cómo funciona?
- **Inferencia Desacoplada (Two-Stage Pipeline)**: En patología computacional moderna, desacoplar la extracción de características de la agregación MIL reduce el tiempo de entrenamiento de días a escasos minutos por época:
  - *Etapa A (Concluida / Pre-calculada)*: Paso por el backbone neuronal (ej. ViT-Base preentrenado con iBOT sobre histología humana) para generar representaciones densas de 768 dimensiones.
  - *Etapa B (Entrenamiento Local / GPU ligera)*: Optimización exclusiva de los pesos del mecanismo de atención MIL ($W, V, U$ en ABMIL/CLAM) sobre las matrices de embeddings de 399 láminas.
- **Foundation Models (iBOT ViT-Base)**: El modelo iBOT (*image BERT pre-training with Online Tokenizer*) entrenado sobre colecciones patológicas (COAD + CAMELYON16) captura correlaciones nucleares complejas y atipia celular con menor dimensionalidad (768 vs 2048 de MoCo), optimizando el uso de memoria GPU.
- **Reproducibilidad y Benchmarking**: Permite validar directamente los resultados reportados en la literatura académica de patología digital alcanzando AUCs superiores a $0.95$ en el conjunto de test independiente.

### 📊 Flujo de Modelado y Transferencia de Embeddings
```text
┌─────────────────────────────────┐     ┌─────────────────────────────────┐
│ Opción iBOT ViT-Base (~4.3 GB)  │     │ Opción MoCo WideResNet (~11.9 GB│
│ 768-dim por parche              │     │ 2048-dim por parche             │
└────────────────┬────────────────┘     └────────────────┬────────────────┘
                 │                                       │
                 └───────────────────┬───────────────────┘
                                     │
                                     ▼ embeddings.h5
┌─────────────────────────────────────────────────────────────────────────┐
│ Partición Experimental (269 Train / 130 Test) según Coords              │
└────────────────────────────────────┬────────────────────────────────────┘
                                     │
                                     ▼
┌─────────────────────────────────────────────────────────────────────────┐
│ Agregadores MIL: AB-MIL  │  CLAM (SB/MB)  │  TransMIL                   │
│ Optimización con Cross-Entropy / Smooth SVM Loss                        │
└────────────────────────────────────┬────────────────────────────────────┘
                                     │
                                     ▼
┌─────────────────────────────────────────────────────────────────────────┐
│ Evaluación Clínica: ROC-AUC > 0.95 en Test Set y Mapas de Atención      │
└─────────────────────────────────────────────────────────────────────────┘
```
---